# Z2005 — Week 11: Introduction to Graphs

Graph terminology, adjacency-matrix and adjacency-list representations, and BFS/DFS traversal — with applications to connected components, cycle detection, and topological sort.

## Learning Objectives

By the end of this notebook you will be able to:

- Define a graph using correct terminology (vertex, edge, directed/undirected, weighted/unweighted, degree) and identify these properties in a given graph.
- Build a graph using both an adjacency matrix and an adjacency list, and explain the space/time trade-offs between the two representations.
- Implement breadth-first search (BFS) and depth-first search (DFS), both iteratively, and DFS recursively, and trace their visit order by hand on a small graph.
- Use BFS to compute connected components and shortest path (in hop count) between two vertices in an unweighted graph.
- Detect cycles in undirected and directed graphs, and produce a valid topological order for a directed acyclic graph (DAG).
- Explain, with a concrete counter-example, why plain BFS hop-count is not the same as shortest path in a *weighted* graph.

## How to use this notebook

Run the cells **top to bottom**. Markdown cells explain a concept; the code cell right after it shows a fully worked, commented example.

Cells marked `# TODO` are for you to complete — replace `raise NotImplementedError` with your implementation. Every exercise is followed by a **Self-Check** cell: it uses `assert` statements, so it silently does nothing (or prints a success message) if your code is correct, and raises an `AssertionError` (or shows exactly which assertion failed) if it is not. Re-run a self-check cell after every change until it passes.

The **Solutions** section at the very end has fully worked answers to every exercise — try each exercise yourself first before looking.

## 1. Graph terminology

A **graph** is a collection of **vertices** (also called nodes) connected by **edges**. That is the entire idea — everything else is vocabulary for describing particular kinds of graphs and what we do with them.

A few distinctions matter a lot in practice:

- **Directed vs. undirected.** In a directed graph, an edge `(u, v)` only lets you go from `u` to `v` — think "Alice follows Bob" on a social network, which does not imply Bob follows Alice. In an undirected graph, an edge is symmetric — think "Alice and Bob are Facebook friends", where the relationship goes both ways automatically.
- **Weighted vs. unweighted.** A weighted graph attaches a number (a *weight* or *cost*) to each edge — for example, the distance in kilometres between two cities, or the toll on a road. An unweighted graph treats every edge as costing the same (effectively weight 1), so "shortest path" just means "fewest edges" (fewest **hops**).
- **Degree.** The degree of a vertex is the number of edges touching it (for directed graphs we usually split this into *in-degree* and *out-degree*). A vertex with degree 0 is disconnected from the rest of the graph.

A common pitfall: students often assume a graph must be "connected" (every vertex reachable from every other), but that is not part of the definition — a graph can have several separate pieces (we will formalize this as **connected components** in Section 5).

The example below builds one small graph that we will reuse throughout this notebook, so that every traversal and algorithm can be traced against the same picture.

In [ ]:
# The running example graph for this notebook:
#
#     0 --- 1 --- 3 --- 4
#      \   /
#       \ /
#        2
#
# Vertices: 0, 1, 2, 3, 4
# Edges (undirected, unweighted): (0,1), (0,2), (1,2), (1,3), (3,4)

NUM_VERTICES = 5
EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (3, 4)]

# Degree of each vertex = how many edges touch it.
# We compute this directly from the edge list, without building a full
# graph structure yet, just to make the definition concrete.
degree = [0] * NUM_VERTICES
for u, v in EDGES:
    degree[u] += 1  # u gains one edge
    degree[v] += 1  # v gains one edge (undirected: both endpoints count)

for vertex in range(NUM_VERTICES):
    print(f"vertex {vertex}: degree {degree[vertex]}")

# Vertex 1 sits on the most edges (0-1, 1-2, 1-3), so it should have the
# highest degree in this small graph.
assert degree[1] == max(degree)
print("\ndegree computation matches the picture above")

## 2. Representing a graph: adjacency matrix vs. adjacency list

Once you know what a graph *is*, you need to decide how to *store* it in memory. There are two standard choices, and picking the wrong one for your situation is a common source of slow code in interviews and in real systems alike.

**Adjacency matrix**: an `n x n` grid where cell `[u][v]` holds the weight of the edge from `u` to `v` (or 0 / `None` if no edge exists). Checking "is there an edge between `u` and `v`?" is a single array lookup — O(1) — regardless of how many edges the graph has. The cost is memory: you pay for `n^2` cells even if the graph has very few edges.

**Adjacency list**: an array of `n` lists, where list `u` holds the neighbors of vertex `u` (and the weight of each edge, if weighted). This uses memory proportional to the actual number of edges, `O(V + E)`, which is much better for **sparse** graphs (few edges relative to `n^2`) — most real-world graphs (road networks, social networks, the web) are sparse. The cost is that checking "is there an edge between `u` and `v`?" now means scanning vertex `u`'s neighbor list, which is O(degree(u)) instead of O(1).

Rule of thumb: adjacency list is the default choice for most problems (it is what you will use for BFS/DFS below); reach for a matrix only when the graph is small and dense, or when you need very frequent O(1) edge-existence checks.

In [ ]:
class GraphMatrix:
    """Adjacency-matrix representation of a graph with n vertices."""

    def __init__(self, n: int):
        self.n = n
        # n x n grid, all zeros = no edges yet. 0 doubles as "no edge"
        # here because we assume all real edge weights are positive.
        self.matrix = [[0] * n for _ in range(n)]

    def add_edge(self, u: int, v: int, weight: int = 1, directed: bool = False) -> None:
        self.matrix[u][v] = weight
        if not directed:
            self.matrix[v][u] = weight  # undirected: mirror the entry

    def has_edge(self, u: int, v: int) -> bool:
        return self.matrix[u][v] != 0  # O(1): direct array lookup


class GraphList:
    """Adjacency-list representation of a graph with n vertices."""

    def __init__(self, n: int):
        self.n = n
        # One empty list of (neighbor, weight) pairs per vertex.
        self.adj = [[] for _ in range(n)]

    def add_edge(self, u: int, v: int, weight: int = 1, directed: bool = False) -> None:
        self.adj[u].append((v, weight))
        if not directed:
            self.adj[v].append((u, weight))  # undirected: add both directions

    def has_edge(self, u: int, v: int) -> bool:
        # O(degree(u)): must scan u's neighbor list, unlike the matrix.
        return any(neighbor == v for neighbor, _weight in self.adj[u])


# Build both representations of the SAME graph from Section 1, and check
# they agree on every possible (u, v) pair.
gm = GraphMatrix(NUM_VERTICES)
gl = GraphList(NUM_VERTICES)
for u, v in EDGES:
    gm.add_edge(u, v)
    gl.add_edge(u, v)

for u in range(NUM_VERTICES):
    for v in range(NUM_VERTICES):
        assert gm.has_edge(u, v) == gl.has_edge(u, v)

print("adjacency matrix:")
for row in gm.matrix:
    print(" ", row)

print("\nadjacency list:")
for vertex, neighbors in enumerate(gl.adj):
    print(f"  {vertex}: {neighbors}")

print("\nboth representations agree on every edge")

### Measuring the space trade-off directly

The paragraph above claims the matrix uses O(n^2) memory regardless of edge count, while the list uses O(V + E). We can check this is not just a theoretical claim by counting actual stored entries for a **sparse** graph (5 vertices, only 5 edges out of a possible 10).

In [ ]:
# Count non-zero cells in the matrix vs. total entries across all
# adjacency lists, for the same sparse graph.
nonzero_matrix_cells = sum(1 for row in gm.matrix for cell in row if cell != 0)
total_list_entries = sum(len(neighbors) for neighbors in gl.adj)

# Both should equal 2 * len(EDGES), since each undirected edge is stored
# twice (once per endpoint) in both representations.
assert nonzero_matrix_cells == total_list_entries == 2 * len(EDGES)

# But the matrix ALSO allocates the zero cells -- that's the wasted space.
total_matrix_cells = NUM_VERTICES * NUM_VERTICES
wasted_cells = total_matrix_cells - nonzero_matrix_cells

print(f"matrix: {total_matrix_cells} cells allocated, only {nonzero_matrix_cells} used ({wasted_cells} wasted)")
print(f"list:   {total_list_entries} entries allocated, all of them used")
print("\nfor a SPARSE graph like this one, the list wastes nothing while the matrix wastes most of its space")

## 3. Breadth-first search (BFS)

BFS explores a graph one "layer" at a time: first the start vertex, then all of its direct neighbors, then all vertices two hops away, and so on. It uses a **queue** (first-in, first-out) to keep track of which vertex to visit next, and a **visited set** so it never processes the same vertex twice (without this, BFS on a graph with a cycle would loop forever).

BFS is the right tool whenever you want the *fewest edges* between two vertices in an unweighted graph — the order BFS visits vertices in is exactly ordered by hop-distance from the start. This is why it is used for things like "degrees of separation" on a social network, or finding the shortest route through a maze where every step costs the same.

A common pitfall: marking a vertex as visited when you *pop* it from the queue, rather than when you *add* it. If you wait until popping, the same vertex can be pushed onto the queue multiple times before it is first processed, wasting work (and in some formulations, breaking correctness).

In [ ]:
from collections import deque

def bfs(adj: dict, start: int) -> list:
    """Breadth-first traversal order starting from `start`.

    adj: dict mapping vertex -> list of neighbor vertices.
    Returns the list of vertices in the order BFS visits them.
    """
    visited = {start}          # mark visited the MOMENT we enqueue, not when we dequeue
    queue = deque([start])     # FIFO queue holding vertices to process
    order = []                 # records the visiting order, for inspection/testing

    while queue:
        node = queue.popleft()   # take the oldest-enqueued vertex
        order.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                visited.add(neighbor)   # mark now, before it can be enqueued again
                queue.append(neighbor)
    return order


# Adjacency-list form as a plain dict, matching the running example graph.
adj = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1], 3: [1, 4], 4: [3]}

bfs_order = bfs(adj, 0)
print("BFS from vertex 0:", bfs_order)

# Starting at 0: layer 0 = {0}, layer 1 = {1, 2} (both direct neighbors of 0),
# layer 2 = {3} (neighbor of 1), layer 3 = {4} (neighbor of 3).
assert bfs_order == [0, 1, 2, 3, 4]
print("matches the expected layer-by-layer order")

## 4. Depth-first search (DFS)

DFS explores as far as possible along one branch before backtracking, using a **stack** (last-in, first-out) instead of a queue. You can implement it two ways: **iteratively**, with an explicit stack (mirroring the BFS code above almost exactly, just swapping the queue for a stack), or **recursively**, letting the language's own call stack do the bookkeeping for you.

DFS is the natural choice whenever you need to explore an entire structure exhaustively rather than find the shortest route — for example, detecting cycles, finding connected components, solving mazes/puzzles with backtracking, or computing a topological order (Section 6).

A common pitfall with the recursive form: for very deep or very large graphs, recursion can hit Python's recursion limit and crash with a `RecursionError`. The iterative form does not have this problem, because it manages its own stack on the heap rather than relying on the call stack.

In [ ]:
def dfs_iterative(adj: dict, start: int) -> list:
    """Depth-first traversal order, using an explicit stack (no recursion)."""
    visited = set()       # here we mark visited on POP, since a vertex may be
    stack = [start]       # pushed multiple times before it is first processed
    order = []

    while stack:
        node = stack.pop()          # take the MOST recently pushed vertex
        if node in visited:
            continue                 # skip if we already processed it via another path
        visited.add(node)
        order.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                stack.append(neighbor)
    return order


def dfs_recursive(adj: dict, start: int) -> list:
    """Depth-first traversal order, using the call stack via recursion."""
    visited = set()
    order = []

    def visit(node: int) -> None:
        visited.add(node)
        order.append(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                visit(neighbor)      # recurse before trying the next neighbor -- this
                                      # is what makes it "depth"-first
    visit(start)
    return order


dfs_iter_order = dfs_iterative(adj, 0)
dfs_rec_order = dfs_recursive(adj, 0)
print("DFS (iterative) from vertex 0:", dfs_iter_order)
print("DFS (recursive) from vertex 0:", dfs_rec_order)

# Both forms visit exactly the same SET of vertices, and both are valid DFS
# orders, but they need not match exactly: the stack (LIFO) pops neighbors
# in the reverse of the order they were pushed, while recursion visits
# neighbors in the order the for-loop encounters them. That is a real,
# easy-to-miss pitfall: "a DFS order" is not unique, it depends on the
# order neighbors are stored in and how you traverse the stack/call stack.
assert set(dfs_iter_order) == set(dfs_rec_order) == set(adj.keys())
assert dfs_iter_order[0] == dfs_rec_order[0] == 0   # both must start at the given start vertex
assert dfs_rec_order == [0, 1, 2, 3, 4]              # recursion follows adj's neighbor order directly
assert dfs_iter_order == [0, 2, 1, 3, 4]             # the stack processes 0's neighbors in reverse
print("\nboth are valid DFS orders over the same vertex set, but the exact order differs")
assert dfs_iter_order != bfs_order

## 5. Connected components

A **connected component** is a maximal set of vertices that can all reach each other. A graph does not have to be a single connected blob — it can be made of several separate "islands", each internally connected but with no edges between islands.

Finding all connected components is a direct application of BFS (or DFS): run a traversal from any unvisited vertex, mark everything it reaches as belonging to the same component, then repeat from the next unvisited vertex, until every vertex has been assigned to exactly one component.

This shows up constantly in practice: finding clusters of mutually-friended users in a social network, detecting which parts of a circuit board are electrically connected, or checking whether a network of pipes/roads is fully connected end-to-end.

In [ ]:
def connected_components(adj: dict, n: int) -> list:
    """Return a list of components; each component is a list of vertices."""
    visited = set()
    components = []
    for vertex in range(n):
        if vertex not in visited:
            # BFS from an unvisited vertex reaches exactly its own component.
            component = bfs(adj, vertex)
            visited.update(component)
            components.append(component)
    return components


# A graph made of three separate "islands": {0,1}, {2,3,4}, and {5} alone.
islands_adj = {0: [1], 1: [0], 2: [3], 3: [2, 4], 4: [3], 5: []}

components = connected_components(islands_adj, 6)
print("connected components:", components)

assert components == [[0, 1], [2, 3, 4], [5]]
print("found exactly the three expected islands, including the isolated vertex 5")

## 6. Cycle detection and topological sort

**Cycle detection in an undirected graph** relies on a simple observation during DFS: if you encounter a neighbor that is already visited *and it is not the vertex you just came from* (its parent in the DFS tree), you have found a back-edge, which means there is a cycle. The "not the parent" check matters because in an undirected graph, every edge you just traversed immediately looks like a "visited neighbor" from the other side — that is not a cycle, just the edge you arrived on.

**Cycle detection in a directed graph** needs a different idea, because the "don't count the parent" trick does not generalize: a directed edge back to an ancestor is a real cycle even though a directed edge to a sibling is not. The standard fix is three-coloring each vertex: **white** (unvisited), **gray** (currently on the DFS recursion stack — an ancestor of the vertex we're at), and **black** (fully finished). A cycle exists exactly when DFS finds an edge into a **gray** vertex.

**Topological sort** only makes sense for a **directed acyclic graph (DAG)** — a directed graph with no cycles, typically representing dependencies (course prerequisites, build steps, task scheduling). It produces an ordering of vertices such that every edge `u -> v` has `u` appearing before `v`. The DFS-based algorithm below runs a full DFS and appends each vertex to the order only when it (and everything reachable from it) is fully explored, then reverses that list.

In [ ]:
def has_cycle_undirected(adj: dict, n: int) -> bool:
    visited = set()

    def dfs_check(node: int, parent: int) -> bool:
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                if dfs_check(neighbor, node):
                    return True
            elif neighbor != parent:   # visited AND not where we came from = a cycle
                return True
        return False

    return any(dfs_check(v, -1) for v in range(n) if v not in visited)


triangle = {0: [1, 2], 1: [0, 2], 2: [0, 1]}   # 0-1-2-0 is a cycle
assert has_cycle_undirected(triangle, 3) is True

no_cycle = {0: [1], 1: [0, 2], 2: [1]}          # a simple path, no cycle
assert has_cycle_undirected(no_cycle, 3) is False
print("undirected cycle detection: both cases correct")


def has_cycle_directed(adj: dict, n: int) -> bool:
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * n

    def dfs_check(node: int) -> bool:
        color[node] = GRAY               # this vertex is now an "ancestor" on our path
        for neighbor in adj[node]:
            if color[neighbor] == GRAY:
                return True               # edge into an ancestor = cycle
            if color[neighbor] == WHITE and dfs_check(neighbor):
                return True
        color[node] = BLACK              # fully finished, no longer an ancestor
        return False

    return any(dfs_check(v) for v in range(n) if color[v] == WHITE)


directed_cycle = {0: [1], 1: [2], 2: [0]}       # 0 -> 1 -> 2 -> 0
assert has_cycle_directed(directed_cycle, 3) is True

course_prereqs = {0: [1, 2], 1: [3], 2: [3], 3: []}   # course 0 unlocks 1 and 2; both unlock 3
assert has_cycle_directed(course_prereqs, 4) is False
print("directed cycle detection: both cases correct")


def topological_sort(adj: dict, n: int) -> list:
    visited = set()
    order = []

    def visit(node: int) -> None:
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                visit(neighbor)
        order.append(node)   # append only once EVERYTHING reachable is done

    for vertex in range(n):
        if vertex not in visited:
            visit(vertex)
    return order[::-1]   # reverse: "finished last" -> "must come first"


topo_order = topological_sort(course_prereqs, 4)
print("topological order of course prerequisites:", topo_order)

# Course 0 must appear before 1 and 2; both 1 and 2 must appear before 3.
position = {course: index for index, course in enumerate(topo_order)}
assert position[0] < position[1] < position[3]
assert position[0] < position[2] < position[3]
print("a valid ordering: 0 is first, 3 is last, exactly as the prerequisites require")

## 7. Where this shows up in practice

- **Social networks**: BFS powers "degrees of separation" / "people you may know" (2nd- and 3rd-degree connections); connected components find friend clusters.
- **Web crawling and the web graph**: DFS/BFS crawl pages reachable from a seed URL; connected components identify isolated sub-webs.
- **Build systems and package managers**: topological sort orders compilation steps or dependency installation so nothing is built before its prerequisites; cycle detection catches circular dependencies (`A` needs `B` needs `A`) before they cause an infinite loop.
- **Maze solving and pathfinding in games**: BFS finds the shortest route (fewest steps) through an unweighted grid maze.

One important limitation to flag before next week: BFS's "fewest hops" answer is only the same as "cheapest route" when every edge costs the same. As soon as edges have different weights (real distances, real costs), BFS can give the wrong answer — which is exactly the gap that Dijkstra's algorithm (coming in the shortest-paths week) is built to close.

In [ ]:
# A quick, concrete demonstration of BFS's blind spot on weighted graphs.
# 0 -> 1 direct edge costs 10; 0 -> 2 -> 1 costs 1 + 1 = 2 (cheaper overall,
# despite being two hops instead of one).
weighted_adj_for_bfs = {0: [1, 2], 1: [0, 2], 2: [0, 1]}  # BFS only sees hop counts
edge_weights = {(0, 1): 10, (1, 0): 10, (0, 2): 1, (2, 0): 1, (1, 2): 1, (2, 1): 1}

bfs_hops_from_0 = bfs(weighted_adj_for_bfs, 0)
# BFS visits 1 before 2 is even relevant to it -- it has no notion of weight
# at all, so it cannot "know" that going via 2 is cheaper.
hop_count_to_1 = bfs_hops_from_0.index(1)   # position in BFS order = hop count

# Real arithmetic, computed from edge_weights directly (not an invented number):
# direct route 0 -> 1 costs edge_weights[(0,1)]; the route via 2 costs
# edge_weights[(0,2)] + edge_weights[(2,1)].
direct_cost = edge_weights[(0, 1)]
via_2_cost = edge_weights[(0, 2)] + edge_weights[(2, 1)]
true_cheapest_cost_to_1 = min(direct_cost, via_2_cost)

print(f"BFS reaches vertex 1 in {hop_count_to_1} hop(s), treating the direct edge as 'closest'")
print(f"but the true cheapest cost to vertex 1 is {true_cheapest_cost_to_1} (via vertex 2), not {edge_weights[(0, 1)]}")
assert hop_count_to_1 == 1          # BFS's answer, by hop count
assert true_cheapest_cost_to_1 == 2  # the actual cheapest total weight
assert true_cheapest_cost_to_1 < edge_weights[(0, 1)]
print("\nconfirmed: BFS's hop-count answer disagrees with the true cheapest-cost answer")

## Exercises

Try each exercise yourself before checking the Solutions section at the end.

### Exercise 1 — Build a flexible `Graph` class

Write a `Graph` class that wraps an adjacency-list dict and supports adding directed or undirected, weighted or unweighted edges, plus a `neighbors(vertex)` method.

```python
g = Graph(5)
g.add_edge(0, 1)
g.add_edge(1, 3, weight=7, directed=True)
g.neighbors(0)   # -> [1]
g.neighbors(1)   # -> [0, 3]   (0 from the undirected edge, 3 from the directed one)
g.neighbors(3)   # -> []       (the 1 -> 3 edge is directed, so 3 has no outgoing edge back)
```

In [ ]:
class Graph:
    """A small adjacency-list graph supporting directed/undirected, weighted edges."""

    def __init__(self, n: int):
        self.n = n
        # TODO: initialize self.adj as a list of n empty lists, one per vertex.
        raise NotImplementedError

    def add_edge(self, u: int, v: int, weight: int = 1, directed: bool = False) -> None:
        # TODO: append (v, weight) to u's neighbor list.
        # TODO: if not directed, also append (u, weight) to v's neighbor list.
        raise NotImplementedError

    def neighbors(self, u: int) -> list:
        # TODO: return a list of just the neighbor vertex numbers for u
        # (drop the weights -- e.g. [(1, 7), (3, 1)] should become [1, 3]).
        raise NotImplementedError

In [ ]:
# Self-Check: Exercise 1
g = Graph(5)
g.add_edge(0, 1)
g.add_edge(1, 3, weight=7, directed=True)

assert g.neighbors(0) == [1]
assert sorted(g.neighbors(1)) == [0, 3]
assert g.neighbors(3) == []   # the 1 -> 3 edge is directed: nothing flows back from 3
assert g.neighbors(4) == []   # vertex 4 has no edges at all

print("Exercise 1 passed")

### Exercise 2 — Shortest hop-distance with BFS

Write `shortest_hop_distance(adj, start, end)` that returns the fewest number of edges needed to get from `start` to `end` in an unweighted graph, or `-1` if `end` is unreachable from `start`.

```python
adj = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1], 3: [1, 4], 4: [3]}
shortest_hop_distance(adj, 0, 4)   # -> 3   (0 -> 1 -> 3 -> 4)
shortest_hop_distance(adj, 0, 0)   # -> 0   (already there)
shortest_hop_distance(adj, 4, 2)   # -> 3   (4 -> 3 -> 1 -> 2)
```

In [ ]:
def shortest_hop_distance(adj: dict, start: int, end: int) -> int:
    """Fewest edges from start to end, or -1 if end is unreachable."""
    # TODO: adapt the bfs() function above to track distance-from-start for
    # each vertex (instead of just visit order), then return dist[end].
    # Return -1 if end never gets a distance assigned (i.e. it's unreachable).
    raise NotImplementedError

In [ ]:
# Self-Check: Exercise 2
adj_ex2 = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1], 3: [1, 4], 4: [3]}

assert shortest_hop_distance(adj_ex2, 0, 4) == 3
assert shortest_hop_distance(adj_ex2, 0, 0) == 0
assert shortest_hop_distance(adj_ex2, 4, 2) == 3

unreachable_adj = {0: [1], 1: [0], 2: []}   # vertex 2 is its own island
assert shortest_hop_distance(unreachable_adj, 0, 2) == -1

print("Exercise 2 passed")

### Exercise 3 — Count connected components

Write `count_components(adj, n)` that returns just the *number* of connected components (an integer), reusing the idea from Section 5 without necessarily reusing the exact function.

```python
adj = {0: [1], 1: [0], 2: [3], 3: [2, 4], 4: [3], 5: []}
count_components(adj, 6)   # -> 3

In [ ]:
def count_components(adj: dict, n: int) -> int:
    """Number of connected components in an undirected graph with n vertices."""
    # TODO: implement using BFS or DFS from every unvisited vertex, counting
    # how many times you start a fresh traversal.
    raise NotImplementedError

In [ ]:
# Self-Check: Exercise 3
adj_ex3 = {0: [1], 1: [0], 2: [3], 3: [2, 4], 4: [3], 5: []}
assert count_components(adj_ex3, 6) == 3

fully_connected = {0: [1, 2], 1: [0, 2], 2: [0, 1]}
assert count_components(fully_connected, 3) == 1

no_edges_at_all = {0: [], 1: [], 2: [], 3: []}
assert count_components(no_edges_at_all, 4) == 4

print("Exercise 3 passed")

### Exercise 4 — Detect a cycle in a directed graph (harder)

Write `find_cycle_directed(adj, n)` that returns a list of vertices forming a cycle if one exists (in the order they appear along the cycle, starting from the vertex where the cycle is first detected), or `None` if the graph is acyclic. This is harder than Section 6's `has_cycle_directed` because you must also *reconstruct* the cycle, not just report that one exists.

Hint: keep a `path` list of the vertices currently on the recursion stack (i.e. currently gray). When you find an edge into a gray vertex, that vertex's position in `path` marks where the cycle begins.

```python
adj = {0: [1], 1: [2], 2: [0]}
find_cycle_directed(adj, 3)   # -> [0, 1, 2]  (some valid rotation of the cycle)

dag = {0: [1, 2], 1: [3], 2: [3], 3: []}
find_cycle_directed(dag, 4)   # -> None
```

In [ ]:
def find_cycle_directed(adj: dict, n: int) -> list:
    """Return a list of vertices forming a cycle, or None if the graph is acyclic."""
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * n
    path = []   # TODO: use this to track vertices currently on the recursion stack

    def dfs_check(node: int):
        # TODO: mark node GRAY and append it to path.
        # TODO: for each neighbor:
        #   - if neighbor is GRAY, a cycle exists: return path[path.index(neighbor):]
        #     (this slices out just the cyclic portion of path)
        #   - if neighbor is WHITE, recurse; if that recursive call finds a
        #     cycle (returns non-None), propagate that result upward immediately
        # TODO: mark node BLACK and pop it from path before returning None
        #   (it's no longer an ancestor of anything once we're done with it)
        raise NotImplementedError

    for vertex in range(n):
        if color[vertex] == WHITE:
            result = dfs_check(vertex)
            if result is not None:
                return result
    return None

In [ ]:
# Self-Check: Exercise 4
cycle_result = find_cycle_directed({0: [1], 1: [2], 2: [0]}, 3)
assert cycle_result is not None
# It should be a rotation of the cycle 0 -> 1 -> 2 -> 0, i.e. 3 distinct
# vertices, each pointing to the next (with the last pointing back to the first).
assert len(cycle_result) == 3
assert set(cycle_result) == {0, 1, 2}

adj_directed_check = {0: [1], 1: [2], 2: [0]}
for i in range(len(cycle_result)):
    current = cycle_result[i]
    following = cycle_result[(i + 1) % len(cycle_result)]
    assert following in adj_directed_check[current]

dag_ex4 = {0: [1, 2], 1: [3], 2: [3], 3: []}
assert find_cycle_directed(dag_ex4, 4) is None

print("Exercise 4 passed")

## Quiz

**1. You need to check for an edge between two vertices thousands of times per second, in a small, dense graph (close to every possible edge exists). Which representation should you pick, and why?**

<details><summary>Show answer</summary>
Adjacency matrix. Edge lookup is O(1) regardless of the graph's density, and since the graph is dense (close to n^2 edges), the matrix's O(n^2) memory cost is not wasted the way it would be on a sparse graph.
</details>

**2. Why does BFS need a "mark as visited when enqueuing" rule, while the iterative DFS shown in this notebook marks as visited when popping instead?**

<details><summary>Show answer</summary>
Both choices are valid ways to avoid infinite loops, but they trade off differently. BFS marks on enqueue so that a vertex is never added to the queue twice, keeping the queue's size bounded by the number of vertices. The iterative DFS shown here marks on pop and instead tolerates a vertex being pushed multiple times, skipping it (via `continue`) if it turns out to already be visited when popped -- a small amount of wasted work in exchange for simpler bookkeeping. Marking on push also works for DFS; both are correct, just different trade-offs.
</details>

**3. What is the smallest structural change needed to detect cycles in a directed graph, given that the undirected "check the parent" trick does not work?**

<details><summary>Show answer</summary>
Track each vertex's state with three colors instead of a simple visited/unvisited flag: WHITE (untouched), GRAY (currently on the DFS recursion stack, i.e. an ancestor of the current vertex), and BLACK (fully finished, no longer an ancestor of anything). A cycle exists exactly when DFS finds an edge leading into a GRAY vertex -- an edge back to a live ancestor.
</details>

**4. Topological sort only works on directed acyclic graphs (DAGs). What goes wrong if you try to run it on a directed graph that contains a cycle?**

<details><summary>Show answer</summary>
There is no valid linear ordering to find: if A must come before B, and B must come before A (a cycle), no ordering can satisfy both requirements simultaneously. In practice, running the DFS-based algorithm on a cyclic graph does not crash, but it silently produces an ordering that violates at least one edge's requirement -- which is why real systems always run cycle detection before trusting a topological sort's output.
</details>

## Solutions (try the exercises yourself first!)

Fully worked solutions to all four exercises below.

In [ ]:
# Solution — Exercise 1
class GraphSolution:
    def __init__(self, n: int):
        self.n = n
        self.adj = [[] for _ in range(n)]

    def add_edge(self, u: int, v: int, weight: int = 1, directed: bool = False) -> None:
        self.adj[u].append((v, weight))
        if not directed:
            self.adj[v].append((u, weight))

    def neighbors(self, u: int) -> list:
        return [neighbor for neighbor, _weight in self.adj[u]]


g_sol = GraphSolution(5)
g_sol.add_edge(0, 1)
g_sol.add_edge(1, 3, weight=7, directed=True)
assert g_sol.neighbors(0) == [1]
assert sorted(g_sol.neighbors(1)) == [0, 3]
assert g_sol.neighbors(3) == []
print("Solution 1 verified")

In [ ]:
# Solution — Exercise 2
def shortest_hop_distance_solution(adj: dict, start: int, end: int) -> int:
    dist = {start: 0}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == end:
            return dist[node]
        for neighbor in adj[node]:
            if neighbor not in dist:
                dist[neighbor] = dist[node] + 1
                queue.append(neighbor)
    return dist.get(end, -1)


adj_sol2 = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1], 3: [1, 4], 4: [3]}
assert shortest_hop_distance_solution(adj_sol2, 0, 4) == 3
assert shortest_hop_distance_solution(adj_sol2, 0, 0) == 0
assert shortest_hop_distance_solution(adj_sol2, 4, 2) == 3
print("Solution 2 verified")

In [ ]:
# Solution — Exercise 3
def count_components_solution(adj: dict, n: int) -> int:
    visited = set()
    count = 0
    for vertex in range(n):
        if vertex not in visited:
            count += 1
            queue = deque([vertex])
            visited.add(vertex)
            while queue:
                node = queue.popleft()
                for neighbor in adj[node]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)
    return count


adj_sol3 = {0: [1], 1: [0], 2: [3], 3: [2, 4], 4: [3], 5: []}
assert count_components_solution(adj_sol3, 6) == 3
print("Solution 3 verified")

In [ ]:
# Solution — Exercise 4
def find_cycle_directed_solution(adj: dict, n: int):
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * n
    path = []

    def dfs_check(node: int):
        color[node] = GRAY
        path.append(node)
        for neighbor in adj[node]:
            if color[neighbor] == GRAY:
                return path[path.index(neighbor):]
            if color[neighbor] == WHITE:
                result = dfs_check(neighbor)
                if result is not None:
                    return result
        color[node] = BLACK
        path.pop()
        return None

    for vertex in range(n):
        if color[vertex] == WHITE:
            result = dfs_check(vertex)
            if result is not None:
                return result
    return None


cycle_sol = find_cycle_directed_solution({0: [1], 1: [2], 2: [0]}, 3)
assert cycle_sol is not None and set(cycle_sol) == {0, 1, 2}
assert find_cycle_directed_solution({0: [1, 2], 1: [3], 2: [3], 3: []}, 4) is None
print("Solution 4 verified")

## MTech Extension — Strongly connected components (Kosaraju's algorithm)

Section 5 covered connected components for **undirected** graphs, where the definition is simple: two vertices are in the same component if any path connects them, in either direction, because undirected edges are already symmetric.

Directed graphs need a stricter notion: a **strongly connected component (SCC)** is a maximal set of vertices where every vertex can reach every other vertex *while respecting edge direction*. Two vertices `u` and `v` are in the same SCC only if there is a directed path from `u` to `v` **and** a directed path from `v` to `u`. A directed graph with no cycles at all (a DAG) has every vertex in its own singleton SCC, since a DAG has no way to "return" to an earlier vertex.

**Kosaraju's algorithm** finds all SCCs in O(V + E) using a neat two-pass trick:

1. Run DFS on the original graph, and record each vertex's *finish time* — the order in which `visit()` returns, exactly like the topological-sort code in Section 6.
2. Build the **transpose graph**: the same vertices, but every edge reversed (`u -> v` becomes `v -> u`).
3. Process vertices in *decreasing* order of finish time from step 1. For each unvisited vertex, run DFS on the **transpose** graph — everything reached in that single DFS call is exactly one SCC.

Why does reversing the graph and using finish-time order work? Intuitively: the vertex that finishes DFS *last* in step 1 is a vertex that nothing else was "waiting on" — it sits at a source-like position with respect to the SCC structure. Running DFS from it on the *reversed* graph can only wander into vertices that could originally reach it, which — combined with it being able to reach them (guaranteed by them appearing in its finish-ordered DFS tree) — is exactly the mutual-reachability that defines an SCC. A full proof is beyond this notebook's scope, but the mechanism is worth being able to reproduce from memory, since it is a common building block (e.g. in compiler dependency analysis and in analyzing web-graph "core" structure).

Compare this to `has_cycle_directed` from Section 6: cycle detection only asks "does at least one cycle exist anywhere?", while SCCs answer the finer-grained question "which *specific* vertices are mutually reachable, and how does the graph decompose into these pieces?" Every cycle lies entirely within a single SCC, but a graph can have zero cycles (a DAG) and still be meaningfully described in terms of its (all-singleton) SCCs — so SCCs are a strict generalization, not just "cycle detection with extra steps."

In [ ]:
def kosaraju_scc(adj: dict, n: int) -> list:
    """Return a list of strongly connected components (each a list of vertices)."""

    # Pass 1: DFS on the original graph, recording finish order (same idea
    # as the DFS-based topological sort in Section 6).
    visited = set()
    finish_order = []

    def dfs_finish(node: int) -> None:
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                dfs_finish(neighbor)
        finish_order.append(node)   # append on the way OUT, i.e. when finished

    for vertex in range(n):
        if vertex not in visited:
            dfs_finish(vertex)

    # Build the transpose graph: reverse every edge's direction.
    transpose = {v: [] for v in range(n)}
    for u in range(n):
        for v in adj[u]:
            transpose[v].append(u)   # edge u -> v becomes v -> u

    # Pass 2: process vertices in DECREASING finish-time order, running DFS
    # on the transpose graph. Each DFS call from an unvisited vertex sweeps
    # out exactly one full SCC.
    visited2 = set()
    components = []
    for vertex in reversed(finish_order):
        if vertex not in visited2:
            stack = [vertex]
            visited2.add(vertex)
            component = []
            while stack:
                node = stack.pop()
                component.append(node)
                for neighbor in transpose[node]:
                    if neighbor not in visited2:
                        visited2.add(neighbor)
                        stack.append(neighbor)
            components.append(component)
    return components


# A graph with two SCCs: {0, 1, 2} form a directed cycle (mutually reachable),
# while {3} sits downstream, reachable FROM the cycle but unable to reach back.
scc_adj = {0: [1], 1: [2], 2: [0, 3], 3: []}

sccs = kosaraju_scc(scc_adj, 4)
print("strongly connected components:", sccs)

# Normalize for comparison: sort vertices within each component, then sort
# components by their smallest vertex, since Kosaraju's exact output order
# depends on DFS visiting order (which is deterministic here, but we compare
# on the sets to make the check robust regardless of dict insertion order).
normalized = sorted([sorted(component) for component in sccs])
assert normalized == [[0, 1, 2], [3]]
print("correctly separates the {0,1,2} cycle from the downstream singleton {3}")

# Sanity check against Section 6's cycle detector: an SCC of size > 1 must
# contain a cycle; a DAG (no cycles at all) must have every SCC be a singleton.
dag_for_scc_check = {0: [1, 2], 1: [3], 2: [3], 3: []}
dag_sccs = kosaraju_scc(dag_for_scc_check, 4)
assert all(len(component) == 1 for component in dag_sccs)
print("on an acyclic graph, every SCC is a singleton, as expected")